In [1]:
import customtkinter as ctk
import pyttsx3
import speech_recognition as sr
import threading
import datetime
import webbrowser
import random

# Initialize CustomTkinter dark theme and tech-cyan accents
ctk.set_appearance_mode("Dark")
ctk.set_default_color_theme("dark-blue")

class AccessibleVoiceAssistantApp(ctk.CTk):
    def __init__(self):
        super().__init__()

        self.title("DeVoice")
        self.geometry("820x720")
        self.resizable(False, False)
        
        # Configure window background for a deep tech look
        self.configure(fg_color="#0D0E12")

        # Initialize Text-to-Speech engine safely
        self.tts_engine = pyttsx3.init()
        self.voices = self.tts_engine.getProperty('voices')
        
        # Default settings
        self.tts_engine.setProperty('rate', 165)
        self.tts_engine.setProperty('volume', 1.0)

        self.create_widgets()

    def create_widgets(self):
        # Header Frame
        header_frame = ctk.CTkFrame(self, fg_color="transparent")
        header_frame.pack(pady=15, fill="x", padx=25)

        self.title_label = ctk.CTkLabel(
            header_frame, 
            text="⚡ De-VOICE INTELLIGENT SUITE", 
            font=ctk.CTkFont(family="Helvetica", size=22, weight="bold"),
            text_color="#00FFCC"
        )
        self.title_label.pack(side="left")

        self.clear_btn = ctk.CTkButton(
            header_frame,
            text="Clear Console",
            width=110,
            height=30,
            command=self.clear_console,
            font=ctk.CTkFont(size=11),
            fg_color="#2A2A2A",
            hover_color="#3A3A3A",
            text_color="#CCCCCC"
        )
        self.clear_btn.pack(side="right")

        # Status / Feedback Indicator with glowing badge styling
        self.status_label = ctk.CTkLabel(
            self, 
            text="● Status: System Ready. Speak or type your command.", 
            font=ctk.CTkFont(family="Arial", size=13, weight="bold"), 
            text_color="#00FF66"
        )
        self.status_label.pack(pady=2)

        # High-Contrast Terminal-style Output Text Box
        self.output_box = ctk.CTkTextbox(
            self, 
            width=760, 
            height=260, 
            font=ctk.CTkFont(family="Courier", size=13), 
            fg_color="#141518", 
            text_color="#00FF66",
            border_color="#00FFCC",
            border_width=1,
            corner_radius=8
        )
        self.output_box.pack(pady=10)
        self.output_box.insert("0.0", "> System initialized successfully.\n> Ready to assist with tasks, searches, and accessibility features.\n")
        self.output_box.configure(state="disabled")

        # Input Frame for Manual/Fallback Typing
        self.input_frame = ctk.CTkFrame(self, fg_color="#18191E", corner_radius=10, border_width=1, border_color="#2A2B32")
        self.input_frame.pack(pady=10, padx=20, fill="x")

        self.entry = ctk.CTkEntry(
            self.input_frame, 
            placeholder_text="Type command (e.g., 'time', 'search python', 'joke')...", 
            width=580, 
            height=45, 
            font=ctk.CTkFont(size=13),
            fg_color="#202126",
            text_color="#FFFFFF",
            border_color="#00FFCC"
        )
        self.entry.pack(side="left", padx=12, pady=10)
        self.entry.bind("<Return>", lambda event: self.process_text_command())

        self.send_btn = ctk.CTkButton(
            self.input_frame, 
            text="Execute", 
            width=120, 
            height=45, 
            command=self.process_text_command, 
            font=ctk.CTkFont(size=13, weight="bold"),
            fg_color="#0083B0",
            hover_color="#005B7F"
        )
        self.send_btn.pack(side="right", padx=12, pady=10)

        # Prominent Voice Interaction Button
        self.voice_btn = ctk.CTkButton(
            self, 
            text="🎙  ACTIVATE VOICE RECOGNITION", 
            width=760, 
            height=50, 
            fg_color="#00FFCC", 
            text_color="#0D0E12",
            hover_color="#00CC99", 
            command=self.listen_to_user, 
            font=ctk.CTkFont(size=15, weight="bold"),
            corner_radius=8
        )
        self.voice_btn.pack(pady=5)

        # Settings Panel (Voice Speed & Preferences)
        settings_frame = ctk.CTkFrame(self, fg_color="transparent")
        settings_frame.pack(pady=5, padx=20, fill="x")

        speed_label = ctk.CTkLabel(settings_frame, text="Speech Speed:", font=ctk.CTkFont(size=11), text_color="#A0A0A0")
        speed_label.pack(side="left", padx=(5, 5))

        self.speed_slider = ctk.CTkSlider(
            settings_frame, from_=100, to=250, number_of_steps=15, 
            width=150, command=self.update_speech_speed
        )
        self.speed_slider.set(165)
        self.speed_slider.pack(side="left", padx=5)

        # Quick Assistant Help Tags
        help_badge = ctk.CTkLabel(
            settings_frame, 
            text="💡 Try saying: 'Time', 'Date', 'Search [query]', 'Joke', or 'Open YouTube'", 
            font=ctk.CTkFont(size=11, slant="italic"), 
            text_color="#6C757D"
        )
        help_badge.pack(side="right", padx=5)

    def speak(self, text):
        """Runs text-to-speech in a safe background thread to prevent UI freezing."""
        def run_tts():
            try:
                self.tts_engine.say(text)
                self.tts_engine.runAndWait()
            except Exception as e:
                print(f"TTS Error: {e}")
        threading.Thread(target=run_tts, daemon=True).start()

    def log_message(self, message):
        """Appends logs safely to the text box and triggers audio feedback."""
        def update_ui():
            self.output_box.configure(state="normal")
            self.output_box.insert("end", f"> {message}\n")
            self.output_box.see("end")
            self.output_box.configure(state="disabled")
        
        self.after(0, update_ui)
        self.speak(message)

    def update_speech_speed(self, value):
        """Updates the speech rate dynamically."""
        self.tts_engine.setProperty('rate', int(value))

    def clear_console(self):
        """Clears the console log window."""
        self.output_box.configure(state="normal")
        self.output_box.delete("0.0", "end")
        self.output_box.insert("0.0", "> Console cleared.\n")
        self.output_box.configure(state="disabled")

    def process_text_command(self):
        """Processes manual text input."""
        cmd = self.entry.get().strip()
        if cmd:
            self.log_message(f"Command received: {cmd}")
            self.entry.delete(0, 'end')
            self.handle_command_logic(cmd)

    def handle_command_logic(self, cmd):
        """Core action handler for user tasks and smart features."""
        lower_cmd = cmd.lower()
        
        if "time" in lower_cmd:
            current_time = datetime.datetime.now().strftime("%I:%M %p")
            self.log_message(f"The current time is {current_time}.")
            
        elif "date" in lower_cmd:
            current_date = datetime.datetime.now().strftime("%A, %B %d, %Y")
            self.log_message(f"Today's date is {current_date}.")
            
        elif "search" in lower_cmd:
            query = lower_cmd.replace("search", "").strip()
            if query:
                self.log_message(f"Searching web for: {query}")
                webbrowser.open(f"https://www.google.com/search?q={query}")
            else:
                self.log_message("Please specify what you would like to search for.")
                
        elif "open youtube" in lower_cmd:
            self.log_message("Opening YouTube.")
            webbrowser.open("https://www.youtube.com")
            
        elif "joke" in lower_cmd:
            jokes = [
                "Why do programmers prefer dark mode? Because light attracts bugs!",
                "There are 10 types of people in the world: those who understand binary, and those who don't.",
                "Algorithm: Word used by programmers when they don't want to explain what they did."
            ]
            self.log_message(random.choice(jokes))
            
        elif "help" in lower_cmd:
            self.log_message("Available commands: time, date, search [query], open youtube, joke, or help.")
            
        else:
            self.log_message(f"Executed action successfully for: {cmd}")

    def listen_to_user(self):
        """Captures microphone input and converts speech to text."""
        def run_recognition():
            r = sr.Recognizer()
            with sr.Microphone() as source:
                self.after(0, lambda: self.status_label.configure(
                    text="● Status: Listening... Speak now.", text_color="#FFCC00"
                ))
                try:
                    r.adjust_for_ambient_noise(source, duration=0.4)
                    audio = r.listen(source, timeout=6, phrase_time_limit=6)
                    
                    self.after(0, lambda: self.status_label.configure(
                        text="● Status: Processing audio...", text_color="#00FFCC"
                    ))
                    text = r.recognize_google(audio)
                    
                    self.after(0, lambda: self.entry.delete(0, 'end'))
                    self.after(0, lambda: self.entry.insert(0, text))
                    self.log_message(f"Recognized: {text}")
                    self.handle_command_logic(text)
                    
                except sr.UnknownValueError:
                    self.log_message("Sorry, I could not understand the audio. Please try again.")
                except sr.RequestError:
                    self.log_message("Speech recognition service is unavailable.")
                except Exception as e:
                    self.log_message(f"Error: {str(e)}")
                finally:
                    self.after(0, lambda: self.status_label.configure(
                        text="● Status: System Ready. Speak or type your command.", text_color="#00FF66"
                    ))

        threading.Thread(target=run_recognition, daemon=True).start()
        self.log_message("Listening for your voice command...")

if __name__ == "__main__":
    app = AccessibleVoiceAssistantApp()
    app.mainloop()

TTS Error: run loop already started
